In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import sys
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
# Add the project root to the sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv
# from langchain.agents import AgentExecutor  # legacy (langchain 0.x; not used — agents come from crunch_llm.code.agents)
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain.agents import create_tool_calling_agent  # legacy (langchain 0.x; not used — agents come from crunch_llm.code.agents)
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from typing import Dict, List, Union, Any 
from pathlib import Path
from IPython.display import display, Image
from typing import Optional,Dict, List, Tuple  
# Encoding
import base64
import mimetypes
import io  
import PIL.Image
import json 
import subprocess
import shutil # 
import re
from datetime import datetime 
import os
from pathlib import Path
# Our Libraries
from crunch_llm.code.agents import *
from crunch_llm.code.teams_SciML_PIML import *
from crunch_llm.code.utils import *

/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/oscar/data/gk/jdtoscan/LLM/Crunch_LLM


# Initialization

In [2]:
my_database_path ="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_PIML"
my_database_path_functions_ollama="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_ollama"
modules_name = 'Layers'
codes_database = 'PINNs_Codes'

In [3]:
## Strategy Team
strategist_agent = memoryAgent(model='gpt-5.1', 
                     verbose=False,
                     system_prompt=strategist_PINN_system_prompt,
                     model_provider='openai',     
                    )
critic_agent = memoryAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=strategist_critic_numerical_prompt_PIML,
                     model_provider='google',     
                    )

## Collection Team
retriever_agent = Retriever_Agent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=Retriever_system_prompt_PIML,
                     db_path=my_database_path,
                     collection_name=codes_database,
                     k=3)
validator_agent = memoryAgent(model='grok-4-1-fast-reasoning', 
                     verbose=False,
                     system_prompt=Validator_system_prompt_PIML,
                     model_provider='grok',     
                    )
collector_agent = Retriever_Agent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=collector_system_prompt_PIML,
                     db_path=my_database_path_functions_ollama,
                     collection_name=modules_name,
                     k=1)

librarian_agent = memoryAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=librarian_system_prompt_PIML,
                     model_provider='google',     
                    )

# Implementation Team
planner_agent = memoryAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=planner_agent_system_prompt_PIML,
                     model_provider='google',     
                    )
planner_parser_agent =  StructuredAgent(
                    model='gpt-oss:120b', 
                    system_prompt=planner_parser_system_prompt,
                    verbose=False
                )
patcher_agent = StructuredAgent(
                    model='gpt-oss:120b', 
                    system_prompt=patcher_agent_system_prompt_PINN,
                    verbose=False
                )

## Debugging Team

inspector_agent = StructuredAgent(
                    model='grok-4-1-fast-reasoning', 
                    model_provider='grok',
                    system_prompt=inspector_system_prompt_PIML,
                    verbose=False
                )


naming_agent = StructuredAgent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=naming_agent_system_prompt_PINN,
                    )


debugger_agent = StructuredAgent(model='claude-sonnet-4-5', 
                     verbose=False,
                     system_prompt=debugger_system_prompt_numerical,
                     model_provider='claude',     
                    )
## Advisor Team
advisor_agent = aRAGAgent_multimodal(
                    model='gemini-3-pro-preview', 
                    verbose=True,
                    system_prompt=advisor_system_prompt_PIML,
                    model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                    db_path=my_database_path_functions_ollama,
                    collection_name=modules_name,
                    k=5
                )
advisor_parser = StructuredAgent(
                    model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=advisor_parser_system_prompt_ON,
                    )

register_agent_parser =StructuredAgent(
                    model='grok-4-1-fast-reasoning', 
                    model_provider='grok',
                    system_prompt=register_agent_system_prompt_numerical,
                    verbose=False
                )


## User Query

In [4]:
user_question = """
Let's solve the 1D Inviscid Burgers' equation:
$$ u_t + u u_x= 0 $$

Domain: $x \in [-1,1]$, $t \in [0, 1]$.
BCs: Periodic in x.

Initial Condition:
$$ u(x,y,0) = -\sin(pi x)$$

Reference Data to compute error metrics:
/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/NM_codes/Burgers_Periodic_SinPiX/2025_11_22_1730/burgers.npz

The code that generated the data saves burgers.npz in NPZ format containing keys: 't' (200,), 'x' (256,), 'usol' (200x256).
"""

In [5]:
checkpoint_project_dir='/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/PINN_codes/Burgers_Periodic_SinPiX/2025_11_24_1434_human/'
strategist_report, error_report, history_log_string, history, initial_code, filename= load_checkpoint_data(checkpoint_project_dir,load_code_only=False)
print(error_report)

--- Checkpoint found. Resuming from: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/PINN_codes/Burgers_Periodic_SinPiX/2025_11_24_1434_human/ ---
--- Loading code from: Burgers_Periodic_SinPiX.py ---
--- Reconstructed 0 run(s) from history log. ---
--- Successfully loaded all checkpoint data. ---
(This is the advisor's report from the *last successful run*)
Advisor Analysis:
Just increase the last stage of SSBroyden the one that has the lowest viscocity. Make sure to train longer in the lowest viscocity regime. For lower bound or minimum viscocity use 1e-4. Divide in more steps 10 at least. Recall that the shortest step must have at least 5000 iterations. We will swich the Foruier embedding by a Periodic wavelet embedding.

$$$$$$\psi(x) = \exp(\kappa \cos(x-\mu)) \times \left[ \kappa \cos(x-\mu) + (\kappa \sin(x-\mu))^2 \right]$$$$$$


Lets try 15 centers first. Recall that this also help us control the periodicity for \kappa= 0.75 check the with the librarian who should write t

In [6]:

max_iterations_all=50
test_image_summary=False
refinement_iterations=0
is_approved = False
base_results_dir='/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/PINN_codes/'
specific_requests=''#'\nSPECIFIC REQUESTS: QR-DeeNet with KKAN and RBF trained for 200000 iterations\n'# Pleaceholder for extra instructions
new_code_details='Test'
clean_functions=''
my_functions=''
requires_name=True
debug_approval=True
critic_approval = False
# --- Initialize history and log string OUTSIDE the loop ---
project_dir = "" # Initialize project_dir
critic_feedback=""
while (is_approved == False and refinement_iterations < max_iterations_all):
        print(f"\n--- Code Creation Iteration: {refinement_iterations} ---")

        # --- Print the history from the LAST completed run ---
        print("\n--- CURRENT TRIAL HISTORY ---")
        print(history_log_string)
        print("---------------------------\n")
        if debug_approval:
                print("--- 1. Calling Strategist Team ---")
                strategist_report, critic_approval=strategy_team(
                                                        user_question,
                                                        strategist_agent,
                                                        critic_agent,
                                                        history_log_string,
                                                        error_report,
                                                        max_iterations=5)
        else:
                print("--- There was an implementation error so we will keep the same strategy for this iteration---")
        if critic_approval:
                if debug_approval:
                        if initial_code=='':
                                print("--- 2. Calling Collection Team (Main files) ---")
                                initial_code, filename, _, _=retrieval_team(
                                                                                strategist_report,
                                                                                retriever_agent,
                                                                                validator_agent,
                                                                                max_iterations= 5)
                                print(f"\n--- Retrieved template: {filename} (First 5 lines) ---")
                                lines = initial_code.split('\n')
                                preview = '\n'.join(lines[:5])
                                print(preview)
                                print("---------------------------------------------------\n")
                        if my_functions=='':
                                print("--- 2.5 Retrieving auxiliary functions ---")

                                my_functions,filename_aux, _, _=retrieval_team_direct(
                                                                                user_question=strategist_report,
                                                                                retriever_agent=collector_agent,
                                                                                max_iterations= 5)
                                print(f"\n--- Retrieved functions: {filename_aux} (First 5 lines) ---")
                                lines = my_functions.split('\n')
                                preview = '\n'.join(lines[:5])
                                print(preview)
                                print("---------------------------------------------------\n")
                        
                        prompt_librarian = (
                        f"[STRATEGY_REPORT]:\n{strategist_report}\n\n"
                        f"[CODE_LIBRARY]:\n{my_functions}\n"
                        )
                        response_librarian=librarian_agent.run(prompt_librarian)
                        clean_functions=response_librarian['output']
                        print(clean_functions)
                print("--- 3. Calling Implementation Team ---")
                filename,initial_code=implementation_team(
                                                                user_question,
                                                                planner_agent,
                                                                patcher_agent,
                                                                planner_parser_agent,
                                                                initial_code,
                                                                clean_functions,
                                                                filename,
                                                                error_report,
                                                                strategist_report,
                                                                remove_comments=False
                                                                )
                print("--- 4. Calling Debugging Team ---")
                prompt_inspector = (
                f"[SCIENTIFIC_STRATEGY_REPORT]:\n{strategist_report}\n\n"
                f"[FINAL_GENERATED_CODE]:\n{initial_code}\n\n"
                )
                response_inspector=inspector_agent.run(prompt_inspector)
                implementation_details=response_inspector['summary']
                generated_data_details=response_inspector['data_saving']
                implementation_approval=check_approval(response_inspector['status'])
                print('Inspector summary:\n',implementation_details)
                if implementation_approval:

                        project_dir,filename,final_filepath,new_code_details= filing_team_PINN(user_question,
                                                                                                filename,
                                                                                                initial_code,
                                                                                                naming_agent,
                                                                                                requires_name,
                                                                                                new_code_details,
                                                                                                base_results_dir=base_results_dir)
                        requires_name=False 
                        success,log_file_name=run_code(project_dir,filename,final_filepath)      
                        status, debugger_report,debug_approval,full_log_for_agent,log_content=debugging_team_PINN(user_question,
                                                                                                        final_filepath,
                                                                                                        log_file_name,
                                                                                                        filename,
                                                                                                        debugger_agent)       

                        if debug_approval:
                                print("--- 5. Calling Advisor Team ---")

                                specific_requests='' 
                                error_report, is_approved, run_summary = advisor_team(
                                                                                user_question=user_question,
                                                                                history_log_string=history_log_string, 
                                                                                project_dir=project_dir,
                                                                                log_content=log_content,
                                                                                strategist_report=strategist_report,
                                                                                implementation_details=implementation_details,
                                                                                advisor_agent=advisor_agent,
                                                                                advisor_parser=advisor_parser,
                                                                                register_agent_parser=register_agent_parser,
                                                                                max_width=800,
                                                                                )
                                run_summary['implementation_details'] = implementation_details
                                run_summary['project_dir'] = project_dir    
                                run_summary['refinement_iterations'] = refinement_iterations       
                                run_summary['data_saving'] = generated_data_details                           
                                history.append(run_summary) 
                                history_log_string = update_and_save_report(
                                        history=history,
                                        user_question=user_question,
                                        strategist_report=strategist_report,
                                        error_report=error_report, # This is the advisor's report
                                        project_dir=project_dir
                                )
                                # if run_summary['change_architecture']:
                                #         print("--- Looking for a new code template---")
                                #         initial_code=''

                
                        else:
                                error_report=f"Debugger Analysis:\n{debugger_report}\n"        
                else:
                        debug_approval=False
                        discrepancies=response_inspector['discrepancies']
                        debugger_report = 'There are discrepancies with the base strategy:\n' + discrepancies     
                        print('Report:',debugger_report)               
                        error_report=f"Debugger Analysis:\n{debugger_report}\n"       
                
        else:
                print('The agents did not get into an agreement')
                break
        print("--- Clearing agent short-term memories ---")
        strategist_agent.clear_history()
        critic_agent.clear_history()
        retriever_agent.clear_history()
        planner_agent.clear_history()
        validator_agent.clear_history()
        advisor_agent.clear_history()
        refinement_iterations=refinement_iterations+1



--- Code Creation Iteration: 0 ---

--- CURRENT TRIAL HISTORY ---
| Run | Model Description | Error | Integrity | Accuracy | Details | Optimality | **Total** |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| 3 | The code implements an MLP architecture with Periodic Fourier Features embedding (kmax=1, sin/cos(k π x) for periodicity on [-1,1]), input as (t, FourierFeatures(x)) without raw x, tanh activations, default width/depth, and hard-constrained output ansatz u(t,x) = -sin(π x) + t * N_θ(t, FourierFeatures(x)) enforcing IC and periodicity. Optimizer workflow is Hybrid Adam + SSBroyden: Phase 1 Adam (fixed ν=0.01, self-balancing λ_pde/λ_ent via grad norm EMA, vRBA weighting inside L_pde, ~10k iterations); Phase 2 SSBroyden (frozen weights from Adam end, vRBA via lambda updates and importance sampling for batches, Hessian reset per ν_k, geometric schedule ν_k = geomspace(0.01, 1e-4, 10 steps), ~10k iterations per ν_k via outer loops x inner Nchange=100). Loss terms: 